# 04 - Validacion cruzada

Evalua la capacidad de generalizacion del MLP usando Stratified K-Fold.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import BATCH_SIZE, DATASET_DIR, METRICS_DIR, EPOCHS, RANDOM_STATE, L2_REGULARIZATION
from src.preprocessing import load_paths, load_split
from src.model import PureTensorFlowMLP, predict, train_mlp
from src.evaluation import compute_metrics

In [ ]:
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
x, y, paths = load_split(DATASET_DIR, 'train')
x.shape, y.shape

In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
rows = []

for fold, (train_idx, val_idx) in enumerate(skf.split(x, y), start=1):
    x_fold_train, y_fold_train = load_paths(
        [paths[index] for index in train_idx],
        y[train_idx],
        augment=True,
    )
    model = PureTensorFlowMLP(input_dim=x.shape[1], seed=RANDOM_STATE + fold)
    train_mlp(
        model,
        x_fold_train,
        y_fold_train,
        x_val=x[val_idx],
        y_val=y[val_idx],
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        l2_strength=L2_REGULARIZATION,
    )
    y_prob = predict(model, x[val_idx])
    rows.append({'fold': fold, **compute_metrics(y[val_idx], y_prob)})

cv_results = pd.DataFrame(rows)
cv_results.to_csv(METRICS_DIR / 'cross_validation_results.csv', index=False)
cv_results

In [ ]:
summary = cv_results.drop(columns=['fold']).agg(['mean', 'std'])
summary.to_csv(METRICS_DIR / 'cross_validation_summary.csv')
summary